In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [3]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from sys import prefix
from listUtils import getFlatList
from musicdb import MusicDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM']
MasterParams()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb


# DB-Specific

In [4]:
from lib import deezer
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath("Deezer")
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Deezer API(Key=None)
Saving Perminant Deezer DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Deezer


# Master Files

In [5]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localRelatedData        = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelatedData".format(db.lower()))
localRelated            = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelated".format(db.lower()))
masterRelatedArtistData = mio.data.getRelatedArtistsData()
localArtistsInfoData    = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfoData".format(db.lower()))
localArtistsInfo        = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfo".format(db.lower()))
masterArtistsInfoData   = mio.data.getArtistsInfoData()
knownArtists            = mio.data.getSummaryNameData()
errors                  = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [6]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Related Artists:       {0}".format(len(localRelated.get())))
print("   Local Related Artists Data:  {0}".format(len(localRelatedData.get())))
print("   Master Related Artists Data: {0}".format(len(masterRelatedArtistData)))
print("   Local Artists Info:          {0}".format(len(localArtistsInfo.get())))
print("   Local Artists Info Data:     {0}".format(len(localArtistsInfoData.get())))
print("   Master Artists Info Data:    {0}".format(masterArtistsInfoData.shape[0]))
print("   Errors:                      {0}".format(len(errors.get())))
print("   Known Summary IDs:           {0}".format(len(knownArtists)))

Deezer Search Results
   Local Related Artists:       93208
   Local Related Artists Data:  0
   Master Related Artists Data: 62431
   Local Artists Info:          874919
   Local Artists Info Data:     0
   Master Artists Info Data:    874874
   Errors:                      1781
   Known Summary IDs:           1069560


# Download Artist Info Data

In [7]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

Deezer API(Key=None)


## Find Artist IDs To Get

In [42]:
artistNames = mio.data.getSummaryNameData()
artistNames.name = "ArtistName"
basicData = DataFrame(artistNames).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
localArtistsInfoDict = localArtistsInfo.get()
artistIDsToGet = basicData[~basicData.index.isin(localArtistsInfoDict.keys())]["ArtistName"]

print("{0} Search Results".format(db))
print("   Known Master Basic Data:   {0}".format(len(artistNames)))
print("   Known Artist Info Names:   {0}".format(len(localArtistsInfoDict)))
print("   Artist Names To Get:       {0}".format(len(artistIDsToGet)))

#   Artist Names To Get:       143303
#   Artist Names To Get:       103403

Deezer Search Results
   Known Master Basic Data:   1069560
   Known Artist Info Names:   995719
   Artist Names To Get:       83303


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistIDsToGet.iteritems()):
    if searchedForLocalArtistsInfo.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalArtistsInfo[artistID] = "NoInfo"
            apiio.sleep(1.5)
        continue
    
    searchedForLocalArtistsInfo[artistID]     = artistName
    searchedForLocalArtistsInfoData[artistID] = response
    apiio.sleep(1.5)
    n += 1
        
    if n % 50 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Process [Getting Deezer ArtistIDs] Start    ==> Time Is 2022-04-08 11:49:39
========================= termTime(day=today,time=9:00pm) =========================
   ====> Terminate Time Set To 2022-04-08 21:00:00 <====
   ====> Will Terminate Process 9 Hours and 10 Minutes From Now
Searching For Artist Info For Johnnie Lee Wills (467938)                        	True
Searching For Artist Info For Gerald Levert and Eddie Levert (261538)           	True
Searching For Artist Info For Magik Matt (12470438)                             	True
Searching For Artist Info For Fisher & Mike Shiver (7354838)                    	True
Searching For Artist Info For Andréa Wood (5259538)                            	True
Searching For Artist Info For New Orleans Jazz Band (1516338)                   	True
Searching For Artist Info For Fernando Cursino (7358338)                        	True
Searching For Artist Info For DJ DUST (6311438)                                 	True
Searching For Artist Info For Z

Searching For Artist Info For Margeaux & Jaakko Manninen (10263338)             	True
Searching For Artist Info For Jack Doe (12250838)                               	True
Searching For Artist Info For Kevin Murray (4646238)                            	True
Searching For Artist Info For Kraut feat. Wasim Arslan (12180038)               	True
Searching For Artist Info For Kumara Yettendra (10095138)                       	True
Searching For Artist Info For Ss'KyrON (1448838)                                	True
Searching For Artist Info For Yann Beuron-Eva Podles-Choeurs de l'Opéra National de Lyon-Orchestre de l'Opéra National de Lyon-Orchestre de Chambre de Grenoble-Marc Minkowski-Sharman Plesner (1498638)	True
Searching For Artist Info For Disco Morato (10944138)                           	True
Searching For Artist Info For Línea Recta (12378638)                           	True
Searching For Artist Info For MI.LA (10767138)                                  	True
Searching For Arti

Searching For Artist Info For VL Deck (8505338)                                 	True
Searching For Artist Info For Wesley Diamond (334338)                           	True
Searching For Artist Info For Tiwadara (10138838)                               	True
Searching For Artist Info For Simona Zagorova (11156338)                        	True
Searching For Artist Info For Mike Anderson (105038)                            	True
Searching For Artist Info For City 3000 (11254938)                              	True
Searching For Artist Info For Джарахов (12371838)                               	True
Searching For Artist Info For Ladybug Transistor (1658638)                      	True
Searching For Artist Info For Rupert Pope, Giles Palmer, Bruce Fingers & Billie Ray Fingers (9732438)	True
Searching For Artist Info For Dawg E. Slaughter (5028338)                       	True
Searching For Artist Info For The Black Star Sound (336338)                     	True
Searching For Artist Info For Rea

Searching For Artist Info For Kim & Reggie Harris, Peter Yarrow & Bethany & Rufus (4586438)	True
Searching For Artist Info For Phutureprimitive, Jillian Ann (4614138)           	True
Searching For Artist Info For Colette Riedinger (794138)                        	True
Searching For Artist Info For Illyricists (7321738)                             	True
250/?      : Process [Getting Deezer ArtistIDs] Has Run For 8 Minutes and 43 Seconds.
Saving 995969 (New=250) Deezer Searched For Artist (Info) IDs
Searching For Artist Info For Coro E Orchestra Gran Teatre Del Liceu, Raimon Torres, Reynald Giovaninetti (4116938)	True
Searching For Artist Info For Dragons (375238)                                  	True
Searching For Artist Info For Kharrington Fairweather (5066138)                 	True
Searching For Artist Info For Blue Motel (9587338)                              	True
Searching For Artist Info For Tim Heintz, Grant Geissman, Charlie Bisharat & Dan Higgins (9425038)	True
Searching For 

Searching For Artist Info For Lucky (58638)                                     	True
Searching For Artist Info For Globotom (7680438)                                	True
Searching For Artist Info For Skinny Joey (5757538)                             	True
Searching For Artist Info For Archer (1224038)                                  	True
Searching For Artist Info For Rohan "Fireball" Richards (4155238)               	True
Searching For Artist Info For Michu MC (10950938)                               	True
Searching For Artist Info For Nora Sagal (5476438)                              	True
Searching For Artist Info For Yung 187 (11499138)                               	True
Searching For Artist Info For Ricky Bombay (5342138)                            	True
Searching For Artist Info For Riho Sibul (7947938)                              	True
Searching For Artist Info For Splacc (12232838)                                 	True
Searching For Artist Info For Darck Rms (11412638)    

...

Searching For Artist Info For Martin A. Hansen (9681936)                        	True
Searching For Artist Info For Dolph (4312836)                                   	True
Searching For Artist Info For Cüneyt Taylan (5608436)                          	True
Searching For Artist Info For Pegasus brozz (4964036)                           	True
Searching For Artist Info For Pepe Pinto con Niño Ricardo (8563936)            	True
Searching For Artist Info For The Bands Of The Royal Air Force (361136)         	True
Searching For Artist Info For The Record's, Serpenti (388436)                   	True
Searching For Artist Info For Get Stellar (1589236)                             	True
Searching For Artist Info For Father Time (1166536)                             	True
Searching For Artist Info For Linoskiii (7269536)                               	True
Searching For Artist Info For Dinner Music Ensemble (4327236)                   	True
Searching For Artist Info For Dario Forzato (8937636) 

Searching For Artist Info For Lila Lindwurm (275236)                            	True
Searching For Artist Info For Gecko's Paradise (9411836)                        	True
Searching For Artist Info For King's College Choir, Cambridge, Gerald Finley, Stephen Cleobury and Timothy Beasley-Murray (11200936)	True
Searching For Artist Info For Radmila Mikic (9838136)                           	True
Searching For Artist Info For Sir David Willcocks-King's College Choir, Cambridge-Ian Hare (4015336)	True
Searching For Artist Info For Lili Poe (7592536)                                	True
Searching For Artist Info For Filero, Grimm, Juan Gotti, Quota (1093636)        	True
Searching For Artist Info For Sadie Chesterfield (8248636)                      	True
Searching For Artist Info For Shailendra Singh, Lata Mangeshkar (873436)        	True
Searching For Artist Info For Bo Gafvert (5284136)                              	True
Searching For Artist Info For Bulgarian Radio Symphony Orchestra, Va

Searching For Artist Info For Marco Moraca (9844136)                            	True
Searching For Artist Info For Sean Boyd (9623036)                               	True
Searching For Artist Info For Muza Rubackyté (1635136)                         	True
4350/?     : Process [Getting Deezer ArtistIDs] Has Run For 2 Hours and 38 Minutes.
Saving 1000069 (New=4350) Deezer Searched For Artist (Info) IDs
Searching For Artist Info For Radikal Dub Kolektiv (102136)                     	True
Searching For Artist Info For Paul Fenoulhet & The Skyrockets Dance Orchestra (7848236)	True
Searching For Artist Info For Mr.Bankroll (12273736)                            	True
Searching For Artist Info For Mensaje Subliminal (6030736)                      	True
Searching For Artist Info For Goldensuns (10345636)                             	True
Searching For Artist Info For Ayub Bacchu (4788036)                             	True
Searching For Artist Info For SveN-R-G vs. Bass-T (9536)               

Searching For Artist Info For JGeek & The Geeks (4588236)                       	True
Searching For Artist Info For Ranqz (9862536)                                   	True
Searching For Artist Info For Cira feat. 2sty, (9958236)                        	True
Searching For Artist Info For Wael Binali, Allan Wilson & The Slovak Radio Symphony (1607336)	True
Searching For Artist Info For Chez n Trent (7850536)                            	True
Searching For Artist Info For Dr. Trincado (9844836)                            	True
Searching For Artist Info For Hypeman (5638936)                                 	True
Searching For Artist Info For Redman Caldwell (5840336)                         	True
Searching For Artist Info For MzK (5967136)                                     	True
Searching For Artist Info For Soulaced (12063036)                               	True
Searching For Artist Info For Sonorama (10067636)                               	True
Searching For Artist Info For Shakemakers

Searching For Artist Info For Barranco, Barrett & Crocker (1289336)             	True
Searching For Artist Info For Luca Binatti & Marc Deeb (6073236)                	True
Searching For Artist Info For Dick Jacobs Chorus (6332336)                      	True
Searching For Artist Info For Frappe Ash (10818036)                             	True
Searching For Artist Info For andrim (182636)                                   	True
Searching For Artist Info For 周迅 (6627236)                                      	True
Searching For Artist Info For Rio Dela Duna & Lizzie Curious (8377436)          	True
Searching For Artist Info For Maribel G. (4151536)                              	True
Searching For Artist Info For Maxi Iborquiza & Niko Van de Moortele (10965036)  	True
Searching For Artist Info For Paul Edelstein (5269936)                          	True
Searching For Artist Info For Ray Gun (12095936)                                	True
Searching For Artist Info For Bernardo Monk (4555536) 

## Save Results

In [39]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [40]:
from pandas import concat
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].fillna(0).astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(len(searchDF[searchDF.index.isin(knownArtists.index)])))
    print("  ==> {0} New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")
    
#Found 995667 Unique Artists

Found 20100 New Artists
Found 975568 Previous Artists
Found 995668 Total Artists
Found 995667 Unique Artists
Found 995667 Unique + Sorted Artists
  ==> 345640 Old Artists
  ==> 650027 New Artists
  ==> 20099 Delta New Artists
Saving Data


In [41]:
localArtistsInfoData.save(data={})

# Download Related Artist Search Data

In [ ]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

## Find Related Artists

In [ ]:
useBasicData       = False
useSelfRelatedData = True
useMasterIDs       = False

if useBasicData is True:
    knownRelatedArtists = localRelated.get()
    basicData = DataFrame(mio.data.getSummaryNameData()).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
    basicData.columns = ["ArtistName", "NumAlbums"]
    artistRelatedToGet = basicData["ArtistName"][~basicData["ArtistName"].index.isin(knownRelatedArtists.keys())]

    print("{0} Search Results".format(db))
    print("   Known Master Basic Data:     {0}".format(basicData.shape[0]))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useSelfRelatedData is True:
    artistRelatedToGet  = {}
    knownRelatedArtists = localRelated.get()
    masterRelatedArtistData = mio.data.getRelatedArtistsData()
    for artistID,artistIDData in masterRelatedArtistData.iteritems():
        artistRelatedToGet.update({str(k): v for k,v in artistIDData.items() if knownRelatedArtists.get(str(k)) is None})
    artistRelatedToGet = Series(artistRelatedToGet)

    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(masterRelatedArtistData)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useMasterIDs is True:
    meu = MusicDBIO()
    mmeDF = meu.getData()
    deezerMatchedArtistsData = mmeDF[mmeDF["Deezer"].notna()][["ArtistName", "Deezer"]]
    deezerMatchedArtists = deezerMatchedArtistsData["ArtistName"].copy(deep=True)
    deezerMatchedArtists.index = deezerMatchedArtistsData["Deezer"]
    artistRelatedToGet = Series({artistID: artistName for artistID,artistName in deezerMatchedArtists.iteritems() if knownRelatedArtists.get(artistID) is None})
    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(deezerMatchedArtists)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "7:36pm")
tt = TermTime("today", "10:15pm")
maxN = 50000000

n  = 0
searchedForLocalRelated         = localRelated.get()
searchedForLocalRelatedData     = localRelatedData.get()
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistRelatedToGet.iteritems()):
    if searchedForLocalRelated.get(artistID) is not None:
        continue

    response = apiio.getArtistRelatedData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(1.5)
        continue
    
    searchedForLocalRelated[artistID]     = artistName
    searchedForLocalRelatedData[artistID] = {str(rec['id']): rec['name'] for rec in response}
    for record in response:
        recID = str(record['id'])
        if searchedForLocalArtistsInfo.get(recID) is None:
            searchedForLocalArtistsInfo[recID]     = artistName
            searchedForLocalArtistsInfoData[recID] = record
    apiio.sleep(1.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
        localRelated.save(data=searchedForLocalRelated)
        localRelatedData.save(data=searchedForLocalRelatedData)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
    
ts.stop()
print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
localRelated.save(data=searchedForLocalRelated)
localRelatedData.save(data=searchedForLocalRelatedData)
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [ ]:
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")

In [ ]:
df = localRelatedData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = mio.data.getRelatedArtistsData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = searchDF.append(Series(df))
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveRelatedArtistsData(data=searchDF)

In [ ]:
localRelatedData.save(data={})
localArtistsInfoData.save(data={})

# Combined Artist Info Data

In [ ]:
artistRelatedData = mio.data.getRelatedArtistsData()
artistRelatedData.name = "RelatedArtists"
artistRelatedData = DataFrame(artistRelatedData)
artistInfoData    = mio.data.getArtistsInfoData()

In [ ]:
artistInfoData

In [ ]:
from pandas import merge
artistInfoData.join(artistRelatedData)

In [ ]:
knownArtists